In [1]:
from dbfread import DBF
import pandas as pd
from pathlib import Path
from itertools import islice

In [2]:
schemas = {}
folder = Path("../dataset/Kfintech AMCs NAVs 09052025")

dbf_files = list(folder.glob("*.dbf"))

In [3]:
for file in dbf_files:
    dbf = DBF(file, encoding="cp1252")

    schema = [
        (field.name, field.type, field.length, field.decimal_count)
        for field in dbf.fields
    ]

    schemas[file.name] = schema

In [4]:
unique_schemas = set(
    tuple(schema)
    for schema in schemas.values()
)

print("DBF files:", len(dbf_files))
print("Unique schemas:", len(unique_schemas))

DBF files: 54
Unique schemas: 1


In [5]:
df = pd.DataFrame(iter(DBF('../dataset/Kfintech AMCs NAVs 09052025/W1N2662.dbf', encoding="cp1252")))

In [6]:
df.shape

(300000, 11)

In [7]:
df.describe()

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
count,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000,300000
unique,4,603,548,622,10233,132824,131923,130752,1,1,524
top,108,BFDP,UTI Gilt Fund - Regular Plan,102BFDP,2015-12-30,10.0000,,,2025-05-11,164331,
freq,167968,2472,3914,2472,133,2457,9715,9715,300000,300000,20228


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   FUND        300000 non-null  str   
 1   SCHEME      300000 non-null  str   
 2   FUNDDESC    300000 non-null  str   
 3   FCODE       300000 non-null  str   
 4   NAVDATE     300000 non-null  object
 5   NAV         300000 non-null  str   
 6   RPOP        300000 non-null  str   
 7   PPOP        300000 non-null  str   
 8   CRDATE      300000 non-null  object
 9   CRTIME      300000 non-null  str   
 10  SCHEMEISIN  300000 non-null  str   
dtypes: object(2), str(9)
memory usage: 56.9+ MB


In [9]:
df.head()

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
0,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-11,12.9885,12.9885,12.9885,2025-05-11,164331,INF789FB1858
1,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-12,12.9904,12.9904,12.9904,2025-05-11,164331,INF789FB1858
2,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-15,12.9961,12.9961,12.9961,2025-05-11,164331,INF789FB1858
3,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-16,10.0001,10.0001,10.0001,2025-05-11,164331,INF789FB1858
4,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-12,10.0000,0.0000,0.0000,2025-05-11,164331,INF789FB1833


In [10]:
fcode_counts = df["FCODE"].value_counts()

print("Number of FCODEs:", fcode_counts.size)
print("\nRecords per FCODE:")
fcode_counts.describe()

Number of FCODEs: 622

Records per FCODE:


count     622.000000
mean      482.315113
std       376.550908
min         1.000000
25%       186.750000
50%       404.000000
75%       792.000000
max      2472.000000
Name: count, dtype: float64

In [11]:
fcode_counts.sort_values(ascending=False).to_frame("RECORDS")

,RECORDS
FCODE,
102BFDP,2472
101TASG,1648
101LISG,1648
102MIMD,1648
108ITGP,1648
...,...
108VGG1,1
108VVQ1,1
108WSG1,1


In [12]:
fcode_dates = (
    df.groupby("FCODE")["NAVDATE"]
      .agg(
          FIRST_DATE="min",
          LAST_DATE="max",
          UNIQUE_DATES="nunique",
          RECORDS="size"
      )
      .sort_values("RECORDS", ascending=False)
)

fcode_dates

,FIRST_DATE,LAST_DATE,UNIQUE_DATES,RECORDS
FCODE,,,,
102BFDP,2001-07-13,2025-04-21,2471,2472
108ULFY,2008-01-11,2024-09-27,1648,1648
101LISG,2015-03-10,2021-12-14,1648,1648
108FSID,2013-01-01,2019-11-07,1648,1648
102BLG1,2013-02-15,2023-03-15,1648,1648
...,...,...,...,...
108WTG1,2016-02-24,2016-02-24,1,1
108T2IG,2008-11-03,2008-11-03,1,1
102N2D1,2014-12-10,2014-12-10,1,1


In [13]:
fcode = df["FCODE"].iloc[4]

fcode_df = df[df["FCODE"] == fcode].copy()

print("FCODE:", fcode)
print("Shape:", fcode_df.shape)

FCODE: 108D1DP
Shape: (76, 11)


In [14]:
fcode_df.head()

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
4,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-12,10.0000,0.0000,0.0000,2025-05-11,164331,INF789FB1833
5,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-16,10.0211,10.0211,10.0211,2025-05-11,164331,INF789FB1833
6,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-19,10.0266,10.0266,10.0266,2025-05-11,164331,INF789FB1833
7,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-20,10.0299,10.0299,10.0299,2025-05-11,164331,INF789FB1833
8,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-21,10.0303,10.0303,10.0303,2025-05-11,164331,INF789FB1833


In [15]:
fcode_df.tail()

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
75,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-05-05,10.1586,10.1586,10.1586,2025-05-11,164331,INF789FB1833
76,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-05-06,10.1170,10.1170,10.1170,2025-05-11,164331,INF789FB1833
77,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-05-07,10.1001,10.1001,10.1001,2025-05-11,164331,INF789FB1833
78,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-05-08,10.1171,10.1171,10.1171,2025-05-11,164331,INF789FB1833
79,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-05-11,10.1389,10.1389,10.1389,2025-05-11,164331,INF789FB1833


In [16]:
fcode_df.nunique(dropna=False)

FUND           1
SCHEME         1
FUNDDESC       1
FCODE          1
NAVDATE       76
NAV           75
RPOP          75
PPOP          75
CRDATE         1
CRTIME         1
SCHEMEISIN     1
dtype: int64

In [17]:
checks = df.groupby("FCODE").agg(
    FUND=("FUND", "nunique"),
    SCHEME=("SCHEME", "nunique"),
    FUNDDESC=("FUNDDESC", "nunique"),
    SCHEMEISIN=("SCHEMEISIN", "nunique")
)

checks[(checks > 1).any(axis=1)]

,FUND,SCHEME,FUNDDESC,SCHEMEISIN
FCODE,,,,
101MCDP,1,1,2,1
101TADP,1,1,2,1
102MFD1,1,1,2,1
102MFDP,1,1,2,1


In [18]:
checks = df.groupby("FCODE").nunique(dropna=False)

problem_fcodes = checks[(checks > 1).any(axis=1)]

print("FCODEs with any repeated/variable column:", len(problem_fcodes))

problem_fcodes

FCODEs with any repeated/variable column: 615


,FUND,SCHEME,FUNDDESC,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
FCODE,,,,,,,,,,
101ETDG,1,1,1,569,546,546,546,1,1,1
101ETDP,1,1,1,255,135,135,135,1,1,1
101FEDD,1,1,1,824,450,450,450,1,1,1
101FRDF,1,1,1,54,49,49,49,1,1,1
101FRDG,1,1,1,770,766,766,766,1,1,1
...,...,...,...,...,...,...,...,...,...,...
RMFEOGP,1,1,1,792,792,791,792,1,1,1
RMFESAG,1,1,1,824,798,798,798,1,1,1
RMFFFAD,1,1,1,501,497,497,497,1,1,1


In [19]:
fcode = "102MFDP"

fcode_df = df[df["FCODE"] == fcode].copy()

print("FCODE:", fcode)
print("Records:", len(fcode_df))

print("\nUnique values per column:")
print(fcode_df.nunique(dropna=False))

fcode_df

FCODE: 102MFDP
Records: 139

Unique values per column:
FUND            1
SCHEME          1
FUNDDESC        2
FCODE           1
NAVDATE       139
NAV           135
RPOP          135
PPOP          136
CRDATE          1
CRTIME          1
SCHEMEISIN      1
dtype: int64


,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
63535,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-12,10.0000,9.9000,10.0000,2025-05-11,164331,INF767K01RP6
63536,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-13,10.0062,9.9061,10.0062,2025-05-11,164331,INF767K01RP6
63537,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW REINV...,102MFDP,2024-12-16,10.0635,9.9629,10.0635,2025-05-11,164331,INF767K01RP6
63538,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-17,9.9915,9.8916,9.9915,2025-05-11,164331,INF767K01RP6
63539,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-18,9.9007,9.8017,9.9007,2025-05-11,164331,INF767K01RP6
...,...,...,...,...,...,...,...,...,...,...,...
66871,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW REINV...,102MFDP,2024-12-05,9.9905,9.8906,9.9905,2025-05-11,164331,INF767K01RP6
66872,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-06,9.9957,9.8957,9.9957,2025-05-11,164331,INF767K01RP6
66873,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-09,9.9830,9.8832,9.9830,2025-05-11,164331,INF767K01RP6
66874,102,MFDP,LIC MF MANUFACTURING FUND - REGULAR IDCW PAYOUT,102MFDP,2024-12-10,10.0124,9.9123,10.0124,2025-05-11,164331,INF767K01RP6


In [20]:
checks.nunique()

FUND            1
SCHEME          1
FUNDDESC        2
NAVDATE       366
NAV           404
RPOP          396
PPOP          400
CRDATE          1
CRTIME          1
SCHEMEISIN      1
dtype: int64

In [21]:
def check_column_types(df):
    results = []

    for col in df.columns:
        s = df[col].dropna().astype(str).str.strip()

        numeric = pd.to_numeric(s, errors="coerce")

        date = pd.to_datetime(
            s,
            format="%Y-%m-%d",
            errors="coerce"
        )

        results.append({
            "COLUMN": col,
            "CURRENT_DTYPE": df[col].dtype,
            "UNIQUE": df[col].nunique(dropna=False),
            "INTEGER": (
                numeric.notna().all()
                and (numeric % 1 == 0).all()
            ),
            "FLOAT": numeric.notna().all(),
            "DATE": date.notna().all(),
            "STRING": True
        })

    return pd.DataFrame(results)

In [22]:
type_check = check_column_types(df)

display(df.head())
display(type_check)

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
0,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-11,12.9885,12.9885,12.9885,2025-05-11,164331,INF789FB1858
1,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-12,12.9904,12.9904,12.9904,2025-05-11,164331,INF789FB1858
2,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-15,12.9961,12.9961,12.9961,2025-05-11,164331,INF789FB1858
3,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-16,10.0001,10.0001,10.0001,2025-05-11,164331,INF789FB1858
4,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-12,10.0000,0.0000,0.0000,2025-05-11,164331,INF789FB1833


,COLUMN,CURRENT_DTYPE,UNIQUE,INTEGER,FLOAT,DATE,STRING
0,FUND,str,4,False,False,False,True
1,SCHEME,str,603,False,False,False,True
2,FUNDDESC,str,548,False,False,False,True
3,FCODE,str,622,False,False,False,True
4,NAVDATE,object,10233,False,False,True,True
5,NAV,str,132824,False,True,False,True
6,RPOP,str,131923,False,False,False,True
7,PPOP,str,130752,False,False,False,True
8,CRDATE,object,1,False,False,True,True
9,CRTIME,str,1,True,True,False,True


In [23]:
dbf_files = list(Path("../dataset/Kfintech AMCs NAVs 09052025/").glob("*.dbf"))

print("DBF files:", len(dbf_files))

DBF files: 54


In [24]:
all_type_checks = []

for file in dbf_files:
    print(f"Checking: {file.name}")

    dbf_data = DBF(file, encoding="cp1252")

    records = list(islice(dbf_data, 2_000))
    df_temp = pd.DataFrame(records)

    result = check_column_types(df_temp)
    result.insert(0, "FILE", file.name)

    all_type_checks.append(result)

type_checks = pd.concat(all_type_checks, ignore_index=True)

display(type_checks)

Checking: W10N8611.dbf
Checking: W11N2151.dbf
Checking: W12N2598.dbf
Checking: W13N339.dbf
Checking: W14N1470.dbf
Checking: W15N4207.dbf
Checking: W16N5742.dbf
Checking: W17N468.dbf
Checking: W18N5916.dbf
Checking: W19N1447.dbf
Checking: W1N2662.dbf
Checking: W20N6789.dbf
Checking: W21N30.dbf
Checking: W22N9069.dbf
Checking: W23N6002.dbf
Checking: W24N6032.dbf
Checking: W25N5867.dbf
Checking: W26N002.dbf
Checking: W27N7735.dbf
Checking: W28N3979.dbf
Checking: W29N5107.dbf
Checking: W2N5399.dbf
Checking: W30N693.dbf
Checking: W31N5788.dbf
Checking: W32N9315.dbf
Checking: W33N4553.dbf
Checking: W34N4780.dbf
Checking: W35N5911.dbf
Checking: W36N746.dbf
Checking: W37N1489.dbf
Checking: W38N1019.dbf
Checking: W39N460.dbf
Checking: W3N5130.dbf
Checking: W40N8909.dbf
Checking: W41N2140.dbf
Checking: W42N4378.dbf
Checking: W43N961.dbf
Checking: W44N4750.dbf
Checking: W45N9081.dbf
Checking: W46N9228.dbf
Checking: W47N668.dbf
Checking: W48N408.dbf
Checking: W49N7630.dbf
Checking: W4N353.dbf
Chec

,FILE,COLUMN,CURRENT_DTYPE,UNIQUE,INTEGER,FLOAT,DATE,STRING
0,W10N8611.dbf,FUND,str,5,False,False,False,True
1,W10N8611.dbf,SCHEME,str,11,False,False,False,True
2,W10N8611.dbf,FUNDDESC,str,10,False,False,False,True
3,W10N8611.dbf,FCODE,str,11,False,False,False,True
4,W10N8611.dbf,NAVDATE,object,1530,False,False,True,True
...,...,...,...,...,...,...,...,...
589,W9N7281.dbf,RPOP,str,1640,False,True,False,True
590,W9N7281.dbf,PPOP,str,1645,False,True,False,True
591,W9N7281.dbf,CRDATE,object,1,False,False,True,True
592,W9N7281.dbf,CRTIME,str,1,True,True,False,True


In [25]:
type_summary = (
    type_checks
    .groupby("COLUMN")
    .agg({
        "CURRENT_DTYPE": "first",
        "INTEGER": "all",
        "FLOAT": "all",
        "DATE": "all",
        "STRING": "all"
    })
)

type_summary

,CURRENT_DTYPE,INTEGER,FLOAT,DATE,STRING
COLUMN,,,,,
CRDATE,object,False,False,True,True
CRTIME,str,True,True,False,True
FCODE,str,False,False,False,True
FUND,str,False,False,False,True
FUNDDESC,str,False,False,False,True
NAV,str,False,True,False,True
NAVDATE,object,False,False,True,True
PPOP,str,False,False,False,True
RPOP,str,False,False,False,True


In [26]:
for col in ["RPOP", "PPOP"]:
    print(f"\n{col}")

    values = set()

    for file in dbf_files:
        dbf_data = DBF(file, encoding="cp1252")

        for i, record in enumerate(dbf_data):
            if i == 2_000:
                break

            value = str(record[col]).strip()

            if value:
                values.add(value)

    numeric = pd.to_numeric(pd.Series(list(values)), errors="coerce")

    print("Unique non-empty values:", len(values))
    print("Non-numeric values:", numeric.isna().sum())
    print("Examples:", pd.Series(list(values))[numeric.isna()].head(20).tolist())


RPOP
Unique non-empty values: 62138
Non-numeric values: 0
Examples: []

PPOP
Unique non-empty values: 62127
Non-numeric values: 0
Examples: []


In [27]:
df_test = df.copy()

df_test["NAVDATE"] = pd.to_datetime(
    df_test["NAVDATE"],
    format="%Y-%m-%d",
    errors="coerce"
)

df_test["NAV"] = pd.to_numeric(
    df_test["NAV"],
    errors="coerce"
)

df_test["RPOP"] = pd.to_numeric(
    df_test["RPOP"],
    errors="coerce"
)

df_test["PPOP"] = pd.to_numeric(
    df_test["PPOP"],
    errors="coerce"
)

df_test["CRDATE"] = pd.to_datetime(
    df_test["CRDATE"],
    format="%Y-%m-%d",
    errors="coerce"
)

df_test["CRTIME"] = pd.to_numeric(
    df_test["CRTIME"],
    errors="coerce"
)

In [28]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype        
---  ------      --------------   -----        
 0   FUND        300000 non-null  str          
 1   SCHEME      300000 non-null  str          
 2   FUNDDESC    300000 non-null  str          
 3   FCODE       300000 non-null  str          
 4   NAVDATE     300000 non-null  datetime64[s]
 5   NAV         300000 non-null  float64      
 6   RPOP        290285 non-null  float64      
 7   PPOP        290285 non-null  float64      
 8   CRDATE      300000 non-null  datetime64[s]
 9   CRTIME      300000 non-null  int64        
 10  SCHEMEISIN  300000 non-null  str          
dtypes: datetime64[s](2), float64(3), int64(1), str(5)
memory usage: 49.2 MB


In [29]:
df_test.head()

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
0,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-11,12.9885,12.9885,12.9885,2025-05-11,164331,INF789FB1858
1,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-12,12.9904,12.9904,12.9904,2025-05-11,164331,INF789FB1858
2,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-15,12.9961,12.9961,12.9961,2025-05-11,164331,INF789FB1858
3,108,D1D1,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1D1,2018-01-16,10.0001,10.0001,10.0001,2025-05-11,164331,INF789FB1858
4,108,D1DP,UTI- DUAL ADVANTAGE FIXED TERM FUND - SERIES I...,108D1DP,2015-01-12,10.0000,0.0000,0.0000,2025-05-11,164331,INF789FB1833


In [30]:
for col in ["NAVDATE", "NAV", "RPOP", "PPOP", "CRDATE", "CRTIME"]:
    print(
        col,
        "original:", df[col].notna().sum(),
        "converted:", df_test[col].notna().sum()
    )

NAVDATE original: 300000 converted: 300000
NAV original: 300000 converted: 300000
RPOP original: 300000 converted: 290285
PPOP original: 300000 converted: 290285
CRDATE original: 300000 converted: 300000
CRTIME original: 300000 converted: 300000


In [31]:
df_test.describe(include="all")

,FUND,SCHEME,FUNDDESC,FCODE,NAVDATE,NAV,RPOP,PPOP,CRDATE,CRTIME,SCHEMEISIN
count,300000,300000,300000,300000,300000,300000.000000,290285.00000,290285.000000,300000,300000.0,300000
unique,4,603,548,622,NaN,NaN,NaN,NaN,NaN,NaN,524
top,108,BFDP,UTI Gilt Fund - Regular Plan,102BFDP,NaN,NaN,NaN,NaN,NaN,NaN,
freq,167968,2472,3914,2472,NaN,NaN,NaN,NaN,NaN,NaN,20228
mean,NaN,NaN,NaN,NaN,2016-03-18 04:22:59,207.608464,210.75120,211.214715,2025-05-11 00:00:00,164331.0,NaN
min,NaN,NaN,NaN,NaN,1991-04-03 00:00:00,0.000000,0.00000,0.000000,2025-05-11 00:00:00,164331.0,NaN
25%,NaN,NaN,NaN,NaN,2013-06-10 00:00:00,10.331775,10.32340,10.360000,2025-05-11 00:00:00,164331.0,NaN
50%,NaN,NaN,NaN,NaN,2016-07-28 00:00:00,11.894600,12.00230,12.042300,2025-05-11 00:00:00,164331.0,NaN
75%,NaN,NaN,NaN,NaN,2020-05-18 00:00:00,21.831850,22.24740,22.427400,2025-05-11 00:00:00,164331.0,NaN
max,NaN,NaN,NaN,NaN,2025-05-11 00:00:00,5159.654700,5159.65470,5159.654700,2025-05-11 00:00:00,164331.0,NaN


In [32]:
for col in ["RPOP", "PPOP"]:
    original = df[col].astype(str).str.strip()
    converted = pd.to_numeric(df[col], errors="coerce")

    empty = original == ""
    nan_after_conversion = converted.isna()

    print(f"\n{col}")
    print("Empty strings:", empty.sum())
    print("NaN after conversion:", nan_after_conversion.sum())
    print("Match:", (empty == nan_after_conversion).all())


RPOP
Empty strings: 9715
NaN after conversion: 9715
Match: True

PPOP
Empty strings: 9715
NaN after conversion: 9715
Match: True


In [33]:
pd.concat(
    [
        df[["FCODE", "RPOP", "PPOP"]].add_suffix("_original"),
        df_test[["RPOP", "PPOP"]].add_suffix("_converted")
    ],
    axis=1
).loc[
    df_test[["RPOP", "PPOP"]].isna().any(axis=1)
].head()

,FCODE_original,RPOP_original,PPOP_original,RPOP_converted,PPOP_converted
19910,102AHDP,,,NaN,NaN
19911,102AHDP,,,NaN,NaN
19912,102AHDP,,,NaN,NaN
19913,102AHDP,,,NaN,NaN
19914,102AHDP,,,NaN,NaN
